In [2]:
import xarray as xr
import pandas as pd
import glob
import os
import math

import numpy as np
import re
from joblib import Parallel, delayed
from pathlib import Path
import matplotlib.pyplot as plt
import alphashape
from itertools import combinations
from collections import defaultdict, deque
from concurrent.futures import ThreadPoolExecutor
from concurrent.futures import ProcessPoolExecutor, as_completed


from shapely import points, contains
import random

In [34]:
from funs.hm_class import (
    meta_one_hot_shot,
    EmulatedDataStorage,
    HistoryMatching)

In [35]:
from funs.sampling_functions import *


In [43]:
%run funs/hm_class.py

In [5]:
working_dir = '/glade/work/qingyuany/camml_re'
case_name = "test"

ppe_para = pd.read_csv("/glade/work/qingyuany/camml_re/v1_b/post_simulations/paras.csv", index_col=0)




In [44]:
test_case = HistoryMatching(working_dir, case_name, ppe_para)
test_case.drop_by_name(["TGCLDLWP", "CLDTOT"])
test_case.drop_by_n_survive(n_survive = 200000)
test_case.update_meta()



In [45]:
test_case.group_para_climatology()
summary2d = test_case.shuffle_vars()

clubb_c2rt                               and microp_aero_wsubi_scale                 :        0
clubb_c2rt                               and zmconv_dmpdz                            :   543529
clubb_c2rt                               and microp_aero_wsub_scale                  :   311843
micro_mg_berg_eff_factor                 and microp_aero_wsub_scale                  :   167056
micro_mg_berg_eff_factor                 and microp_aero_wsubi_scale                 :   108468
clubb_C8                                 and microp_aero_wsub_scale                  :   289692
clubb_C8                                 and microp_aero_wsubi_scale                 :   514619
microp_aero_wsub_scale                   and microp_aero_wsubi_scale                 :   338409
micro_mg_dcs                             and microp_aero_wsubi_scale                 :    91318
microp_aero_wsub_scale                   and microp_aero_npccn_scale                 :   353752
zmconv_dmpdz                            

In [46]:
list(summary2d.values())[0]

,var1,var2,count
192,FLUT_zonal_65to70,LWCF_zonal_70to75,0
1567,FLUT_zonal_60to65,LWCF_zonal_70to75,0
2391,LWCF_zonal_70to75,FLUT_zonal_-50to-45,4567
2076,LWCF_zonal_-30to-25,FLUT_zonal_-50to-45,5063
1389,FLUT_zonal_55to60,LWCF_zonal_70to75,16253
...,...,...,...
2202,SWCF_zonal_20to25,PRECT_zonal_-30to-25,973817
1052,FSNTOA_zonal_-15to-10,FSNTOA_zonal_-10to-5,977886
1029,FSNTOA_zonal_-15to-10,SWCF_zonal_20to25,978684
2207,SWCF_zonal_20to25,FSNTOA_zonal_-10to-5,980890


In [47]:
no_overlap_2d_vars = ['FLUT_zonal_65to70', 'LWCF_zonal_70to75', 'FLUT_zonal_-50to-45', 'FLUT_zonal_55to60']

In [48]:
test_case.drop_no_overlap2d_vars(no_overlap_2d_vars)
test_case.group_para_climatology()
summary2d = test_case.shuffle_vars()
summary2d

clubb_c2rt                               and microp_aero_wsubi_scale                 :    35946
clubb_c2rt                               and zmconv_dmpdz                            :   543529
clubb_c2rt                               and microp_aero_wsub_scale                  :   311843
micro_mg_berg_eff_factor                 and microp_aero_wsub_scale                  :   167056
micro_mg_berg_eff_factor                 and microp_aero_wsubi_scale                 :   108468
clubb_C8                                 and microp_aero_wsub_scale                  :   289692
clubb_C8                                 and microp_aero_wsubi_scale                 :   514619
microp_aero_wsub_scale                   and microp_aero_wsubi_scale                 :   338409
micro_mg_dcs                             and microp_aero_wsubi_scale                 :    91318
microp_aero_wsub_scale                   and microp_aero_npccn_scale                 :   353752
zmconv_dmpdz                            

{}

In [49]:
test_case.paras_vars

{('clubb_c2rt', 'microp_aero_wsubi_scale'): ['FLUT_zonal_45to50',
  'FLUT_zonal_50to55',
  'LWCF_zonal_0to5',
  'LWCF_zonal_-35to-30',
  'FSNTOA_zonal_15to20',
  'LWCF_zonal_-15to-10',
  'LWCF_zonal_50to55',
  'LWCF_zonal_30to35',
  'LWCF_zonal_-45to-40',
  'LWCF_zonal_40to45',
  'FLUT_zonal_5to10',
  'FSNTOA_zonal_0to5',
  'LWCF_zonal_-5to0',
  'FLUT_zonal_-10to-5',
  'FSNTOA_zonal_-5to0',
  'FSNTOA_zonal_-15to-10',
  'FLUT_zonal_0to5',
  'FLUT_zonal_-5to0',
  'FLUT_zonal_30to35',
  'TMQ_zonal_0to5',
  'SWCF_zonal_-15to-10',
  'FLUT_zonal_25to30',
  'FLUT_zonal_35to40',
  'FSNTOA_zonal_5to10',
  'LWCF_zonal_20to25',
  'FLUT_zonal_60to65',
  'LWCF_zonal_-10to-5',
  'FLUT_zonal_-40to-35',
  'TMQ_zonal_5to10',
  'LWCF_zonal_45to50',
  'SWCF_zonal_-10to-5',
  'LWCF_zonal_-25to-20',
  'FLUT_zonal_-20to-15',
  'LWCF_zonal_35to40',
  'LWCF_zonal_-20to-15',
  'FSNTOA_zonal_-20to-15',
  'SWCF_zonal_15to20',
  'FLUT_zonal_-15to-10',
  'FLUT_zonal_20to25',
  'LWCF_zonal_-30to-25',
  'FLUT_zonal_

In [50]:
test_case.drop_by_nvar_per_pair(n_var_thre=1)

In [51]:
test_case.dropped_vars.local

[['PRECT_zonal_0to5'],
 ['PRECT_zonal_-20to-15'],
 ['PRECT_zonal_15to20'],
 ['PRECT_zonal_-60to-55'],
 ['precip095'],
 ['PRECT_zonal_-5to0'],
 ['precip090'],
 ['FSNTOA_zonal_-75to-70']]

In [52]:
test_case.paras_vars

{('clubb_c2rt', 'microp_aero_wsubi_scale'): ['FLUT_zonal_45to50',
  'FLUT_zonal_50to55',
  'LWCF_zonal_0to5',
  'LWCF_zonal_-35to-30',
  'FSNTOA_zonal_15to20',
  'LWCF_zonal_-15to-10',
  'LWCF_zonal_50to55',
  'LWCF_zonal_30to35',
  'LWCF_zonal_-45to-40',
  'LWCF_zonal_40to45',
  'FLUT_zonal_5to10',
  'FSNTOA_zonal_0to5',
  'LWCF_zonal_-5to0',
  'FLUT_zonal_-10to-5',
  'FSNTOA_zonal_-5to0',
  'FSNTOA_zonal_-15to-10',
  'FLUT_zonal_0to5',
  'FLUT_zonal_-5to0',
  'FLUT_zonal_30to35',
  'TMQ_zonal_0to5',
  'SWCF_zonal_-15to-10',
  'FLUT_zonal_25to30',
  'FLUT_zonal_35to40',
  'FSNTOA_zonal_5to10',
  'LWCF_zonal_20to25',
  'FLUT_zonal_60to65',
  'LWCF_zonal_-10to-5',
  'FLUT_zonal_-40to-35',
  'TMQ_zonal_5to10',
  'LWCF_zonal_45to50',
  'SWCF_zonal_-10to-5',
  'LWCF_zonal_-25to-20',
  'FLUT_zonal_-20to-15',
  'LWCF_zonal_35to40',
  'LWCF_zonal_-20to-15',
  'FSNTOA_zonal_-20to-15',
  'SWCF_zonal_15to20',
  'FLUT_zonal_-15to-10',
  'FLUT_zonal_20to25',
  'LWCF_zonal_-30to-25',
  'FLUT_zonal_

In [53]:
test_case.build_hulls()

In [54]:
%run ./funs/sampling_functions.py

In [55]:

para_seq = list(test_case.grouped_hulls.keys())
check = orchestrate_test(para_seq, test_case.p_emu, test_case.tf_masks,  
                         test_case.para_nm, test_case.grouped_hulls, test_case.paras_vars, n_pts=  10000, n_threshold = 100,sample_threshold = 10**5, max_workers = 31)

## Add random seed generator for the batch sampling method

Running ('clubb_c2rt', 'microp_aero_wsubi_scale'), the 0th simulation
There is overlap for ('clubb_c2rt', 'microp_aero_wsubi_scale'). Proceed to the next parameter pair
Running ('clubb_c2rt', 'zmconv_dmpdz'), the 1th simulation
There is overlap for ('clubb_c2rt', 'zmconv_dmpdz'). Proceed to the next parameter pair
Running ('clubb_c2rt', 'microp_aero_wsub_scale'), the 2th simulation
There is overlap for ('clubb_c2rt', 'microp_aero_wsub_scale'). Proceed to the next parameter pair
Running ('micro_mg_berg_eff_factor', 'microp_aero_wsub_scale'), the 3th simulation
There is overlap for ('micro_mg_berg_eff_factor', 'microp_aero_wsub_scale'). Proceed to the next parameter pair
Running ('micro_mg_berg_eff_factor', 'microp_aero_wsubi_scale'), the 4th simulation
Find nothing from 100000 pts
Find nothing, try to resolve it by breaking the variables into groups
First sample out_prev that needs greater sample size, which will take long
The size of out_prev is 1818
	 	Running test to see if ('micro_m

In [56]:
pts = sample_from_hulls_n(list(check[2].keys()), test_case.para_nm, check[2],n_pts=50000, n_threshold=5000,sample_threshold=10**8,max_workers=32)


In [57]:
pts.shape

(729, 34)

In [61]:
test_case.rescale_para(pts)

[('clubb_c2rt', 'microp_aero_wsubi_scale'),
 ('clubb_c2rt', 'zmconv_dmpdz'),
 ('clubb_c2rt', 'microp_aero_wsub_scale'),
 ('micro_mg_berg_eff_factor', 'microp_aero_wsub_scale'),
 ('micro_mg_berg_eff_factor', 'microp_aero_wsubi_scale'),
 ('clubb_C8', 'microp_aero_wsub_scale'),
 ('clubb_C8', 'microp_aero_wsubi_scale'),
 ('microp_aero_wsub_scale', 'microp_aero_wsubi_scale'),
 ('micro_mg_dcs', 'microp_aero_wsubi_scale'),
 ('microp_aero_wsub_scale', 'microp_aero_npccn_scale'),
 ('zmconv_dmpdz', 'microp_aero_wsubi_scale'),
 ('clubb_C8', 'clubb_c2rt')]